# 03 — Feature Engineering

**Workstream**: Modeling (features)  ·  **Owners**: Bella + Deepak (backup: Jun)  ·  **Last touched**: 2026-06-02

**What this notebook decides**

Take the labeled inspections produced by notebook 02 and build the
**modelling table** — one row per inspection anchor, every column either a
feature or a key/label. Output: `data/processed/features.parquet`.

**The single most important property: leak-freeness.** Every `prior_*`
column summarises only data with strictly earlier dates at the same
license. Every `n_311_*` column uses a right-exclusive lookback window. If
any feature ever leaks the anchor's own outcome into its own row, our
evaluation numbers will look amazing on the holdout set and will fall apart
in production. The tests in `tests/test_features.py` exist precisely to
stop this from happening — they assert leak-freeness on synthetic data
where the right answer is known.

**Why this notebook is short.** All the feature logic lives in modules
under `src/foodsafety/features/`, one module per feature family:

  | Module                       | Adds columns                  | Leak guard |
  | ---                          | ---                           | --- |
  | `inspection_features.py`     | `prior_*`, `days_since_*`     | cumsum-minus-self, ffill-shift |
  | `license_features.py`        | `static_*`                    | values are time-invariant |
  | `keyword_flags.py`           | `flag_kw_*`                   | flag is for the anchor's text (not a future leak) |
  | `complaint_features.py`      | `n_311_*_zip_*d`              | right-exclusive rolling window |
  | `build.py`                   | orchestrator + row filter     | drops burn-in / invalid licenses / non-modelable at the END |

This notebook just calls the orchestrator, validates, and writes.

**Contract**: `docs/interface_contracts.md` § 2. **Scope**: `CLAUDE.md`.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

_PROJECT_ROOT = Path.cwd().parent
if str(_PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT / 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from foodsafety.config import PROCESSED_DIR, RAW_DIR
from foodsafety.features.build import build_features

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 220)
plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

## 2. Load the inputs

- `inspections_labeled.parquet` from notebook 02 — the per-anchor table with the label
- `complaints_311.parquet` from notebook 01 — the 311 history used for the
  ZIP-level complaint features

In [ ]:
labeled_path = PROCESSED_DIR / 'inspections_labeled.parquet'
complaints_path = RAW_DIR / 'complaints_311.parquet'
licenses_hist_path = RAW_DIR / 'licenses_historical.parquet'

for p in (labeled_path, complaints_path, licenses_hist_path):
    if not p.exists():
        raise SystemExit(
            f'Missing {p}. Run the earlier notebooks first '
            f'(01 for raw datasets, 02 for labels).'
        )

labeled = pd.read_parquet(labeled_path)
complaints = pd.read_parquet(complaints_path)
licenses_historical = pd.read_parquet(licenses_hist_path)

print(f'labeled:             {len(labeled):,} rows × {labeled.shape[1]} cols  ({labeled_path.stat().st_size/1e6:.1f} MB)')
print(f'complaints:          {len(complaints):,} rows × {complaints.shape[1]} cols  ({complaints_path.stat().st_size/1e6:.1f} MB)')
print(f'licenses_historical: {len(licenses_historical):,} rows × {licenses_historical.shape[1]} cols  ({licenses_hist_path.stat().st_size/1e6:.1f} MB)')
print()
print(f'labeled date range:    {labeled["inspection_date"].min().date()} → {labeled["inspection_date"].max().date()}')
complaints['created_date'] = pd.to_datetime(complaints['created_date'])
print(f'complaints date range: {complaints["created_date"].min().date()} → {complaints["created_date"].max().date()}')

## 3. Run the feature build

Wall-clock: ~30-90 s. The slow steps are the rolling-by-zip merge_asof joins
for the 311 features (one per sr_type). Everything else is sub-second.

**Output row filter** (applied at the end, see `build.py`):

- `is_burnin == False` (kept for prior_* computation, dropped from output)
- `license_id not in {'', '0'}` (placeholder values pool unrelated facilities)
- `results in {'Pass', 'Pass w/ Conditions', 'Fail'}` (drop operational non-outcomes)

In [ ]:
%%time
features = build_features(
    labeled,
    complaints=complaints,
    licenses_historical=licenses_historical,
)
print(f'\nshape: {features.shape[0]:,} rows × {features.shape[1]} cols')

## 4. What did we build?

Group the columns by family for a quick eye-test. The keys + label appear
first, then `prior_*`, then `static_*`, then `flag_kw_*`, then `n_311_*`.

In [ ]:
families = {
    'keys / label':       ['license_id', 'inspection_id', 'inspection_date', 'as_of_date',
                            'y_fail_or_critical_next_180d', 'right_truncated'],
    'prior_* (history)':   sorted(c for c in features.columns if c.startswith('prior_')),
    'days_since_*':        sorted(c for c in features.columns if c.startswith('days_since_')),
    'static_* (categorical)': sorted(c for c in features.columns if c.startswith('static_')),
    'flag_kw_* (NLP layer B)': sorted(c for c in features.columns if c.startswith('flag_kw_')),
    'n_311_* (complaint density)': sorted(c for c in features.columns if c.startswith('n_311_')),
}
for name, cols in families.items():
    print(f'{name} ({len(cols)}):')
    for c in cols:
        print(f'  · {c}')
    print()

## 5. Validate

### 5a. Leak-free spot-check on real data

If the cumsum-minus-self pattern is correct, the anchor's own outcome
shouldn't be reachable from its own features. We can't prove that on real
data — that's what `tests/test_features.py` is for — but we can sanity-check:

- For **first inspection** at each license: `prior_inspections == 0`,
  `days_since_last_inspection` is NaN.
- For **Fail anchors**: `prior_fails` does NOT count this anchor's Fail.
  (We compare to `prior_fail_or_priority_events` which is also leak-free.)

In [ ]:
# Sanity check 1: first inspection at a license should have prior_inspections == 0
first_per_license = features.sort_values('inspection_date').drop_duplicates('license_id', keep='first')
print(f'First-anchor rows: {len(first_per_license):,}')
n_zero_priors = (first_per_license['prior_inspections'] == 0).sum()
print(f'  prior_inspections == 0:        {n_zero_priors:,}  '
      f'({n_zero_priors / len(first_per_license):.1%})')
n_nan_days = first_per_license['days_since_last_inspection'].isna().sum()
print(f'  days_since_last_inspection NaN: {n_nan_days:,}  '
      f'({n_nan_days / len(first_per_license):.1%})')
print()
print('Caveat: not all first-anchors have prior_inspections == 0 because some')
print('licenses have BURN-IN inspections before their first post-2019 anchor.')
print('That is the expected behaviour and the whole point of the burn-in design.')

### 5b. Distribution of `prior_fails`

Expect a long-tailed distribution centred near 0 (most facilities never have a fail), with the tail picking out the chronic offenders.

In [ ]:
print(features['prior_fails'].describe(percentiles=[0.5, 0.9, 0.95, 0.99]).round(2))
ax = features['prior_fails'].clip(upper=10).plot.hist(
    bins=11, title='prior_fails (clipped at 10)')
ax.set_xlabel('prior_fails'); plt.tight_layout(); plt.show()

### 5c. Univariate sanity — does `prior_fails` correlate with the label?

If this isn't monotonic-increasing we have a problem. A facility with 3 prior
fails should be meaningfully more likely to fail again in the next 180 days
than one with 0.

In [ ]:
label = 'y_fail_or_critical_next_180d'
trainable = features.dropna(subset=[label]).copy()
by_prior_fails = (
    trainable.assign(pf_bucket=trainable['prior_fails'].clip(upper=4))
    .groupby('pf_bucket')[label]
    .agg(n='size', positive_rate='mean')
    .round(3)
)
print('Label rate by prior_fails bucket (0..4+):')
print(by_prior_fails)

### 5d. Keyword flag prevalence

Each flag should fire on a non-trivial minority of rows. If one fires on 90%+ 
or on 0%, the regex needs tuning.

In [ ]:
flag_cols = sorted(c for c in features.columns if c.startswith('flag_kw_'))
flag_summary = pd.DataFrame({
    'flag': flag_cols,
    'fire_rate': [features[c].mean() for c in flag_cols],
    'label_rate_when_set': [
        trainable.loc[trainable[c], label].mean() if trainable[c].any() else float('nan')
        for c in flag_cols
    ],
}).sort_values('fire_rate', ascending=False)
print(flag_summary.round(3).to_string(index=False))

### 5e. 311 complaint features — sanity by ZIP

Pick a ZIP with many 311 events and confirm the per-anchor count looks reasonable. If `n_311_rodent_zip_90d` is always 0 or always 100+, something's off.

In [ ]:
complaint_cols = sorted(c for c in features.columns if c.startswith('n_311_'))
print('Complaint-feature summary (across all anchors):')
print(features[complaint_cols].describe(percentiles=[0.5, 0.9, 0.99]).round(2).to_string())

## 6. Write the contract artifact

Output: `data/processed/features.parquet`. The Phase 4b baseline notebook
(`04_baseline_logreg.ipynb`) reads from here.

In [ ]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
out_path = PROCESSED_DIR / 'features.parquet'

# Cast string-typed object cols to pandas string for consistent parquet round-trip.
to_write = features.copy()
obj_cols = to_write.select_dtypes('object').columns
to_write[obj_cols] = to_write[obj_cols].astype('string')

to_write.to_parquet(out_path, index=False)
size_mb = out_path.stat().st_size / 1e6
print(f'wrote → {out_path}')
print(f'        {len(to_write):,} rows · {to_write.shape[1]} cols · {size_mb:.1f} MB')
print()
print(f'Trainable rows (non-NA label): {trainable.shape[0]:,}')
print(f'Positive rate:                 {trainable[label].mean():.2%}')
print(f'Feature columns:               {sum(1 for c in features.columns if c.startswith(("prior_","days_since_","static_","flag_kw_","n_311_")))}')

## 7. Hand-off

**Output**: `data/processed/features.parquet`

**Schema**: see `docs/interface_contracts.md` § 2.

**Next step**: notebook `04_baseline_logreg.ipynb` trains the logistic-regression
baseline against a chronological train/val/test split. The label is
`y_fail_or_critical_next_180d`. Use the temporal splitter (it will live in
`src/foodsafety/utils/time.py`) — never `train_test_split(shuffle=True)`.

**Sanity to confirm before moving on**:

- `tests/test_features.py` passes — leak-free guarantees are still in force
- Label rate climbs monotonically with `prior_fails` bucket (§ 5c)
- Every `flag_kw_*` fires on >0% and <90% of rows (§ 5d)
- Complaint features have a long-tailed but non-zero distribution (§ 5e)